# 04 - putting it all together

so far we've got:
- `train_v2.csv` - the labels (who churned)
- `members_v3.csv` - user demographics
- `user_logs_v2_agg.parquet` - listening summary per user, from notebook 02
- `transactions_v2_agg.parquet` - payment summary per user, from notebook 03

time to join all of these into one table - one row per user with everything we know about them, plus the churn label. this is the table we'll actually train a model on later.


## Step 1: load everything

just loading all four pieces we need.

- `train` and `members` come straight from the raw csvs, same as notebook 01
- the other two are the parquet files we saved earlier - `pd.read_parquet` reads those back instantly


In [2]:
import pandas as pd
from pathlib import Path

DATA = Path("../data/raw")
PROCESSED = Path("../data/processed")

train = pd.read_csv(DATA / "train_v2.csv")
members = pd.read_csv(DATA / "members_v3.csv")
user_logs_agg = pd.read_parquet(PROCESSED / "user_logs_v2_agg.parquet")
transactions_agg = pd.read_parquet(PROCESSED / "transactions_v2_agg.parquet")

print("train:            ", train.shape)
print("members:          ", members.shape)
print("user_logs_agg:    ", user_logs_agg.shape)
print("transactions_agg: ", transactions_agg.shape)


train:             (970960, 2)
members:           (6769473, 6)
user_logs_agg:     (1103894, 10)
transactions_agg:  (1197050, 8)


## Step 2: merge everything onto train

`train` is our base - it has exactly the users we need to predict for. we'll attach the other tables onto it one by one, matching rows up by `msno`.

- `.merge(other, on='msno', how='left')` - for every row in the left table, look for a matching `msno` in `other` and bring its columns along
- `how='left'` means: keep every row from the left side no matter what. if there's no match in `other`, those new columns just become `NaN` for that user
- doing this 3 times in a row, chaining onto `df` each time

after all, `df` should still have the same number of rows as `train` (970,960) - we're just adding columns, not adding or removing users


In [3]:
df = train.merge(members, on='msno', how='left')
df = df.merge(transactions_agg, on='msno', how='left')
df = df.merge(user_logs_agg, on='msno', how='left')

print(df.shape)
df.head()


(970960, 23)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,num_transactions,total_actual_paid,total_plan_list_price,...,last_expire_date,num_days,num_25_sum,num_50_sum,num_75_sum,num_985_sum,num_100_sum,num_unq_sum,total_secs_sum,total_secs_mean
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5.0,28.0,male,3.0,20131223.0,NaN,NaN,NaN,...,NaT,11.0,186.0,23.0,13.0,10.0,318.0,348.0,80598.557,7327.141545
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13.0,20.0,male,3.0,20131223.0,1.0,180.0,180.0,...,2017-04-11,6.0,0.0,4.0,2.0,0.0,26.0,30.0,6986.509,1164.418167
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13.0,18.0,male,3.0,20131227.0,2.0,300.0,300.0,...,2017-06-15,20.0,239.0,57.0,32.0,22.0,205.0,432.0,67810.467,3390.523350
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1.0,0.0,NaN,7.0,20140109.0,10.0,1490.0,1490.0,...,2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,13.0,35.0,female,7.0,20140125.0,8.0,792.0,792.0,...,2017-09-18,15.0,9.0,7.0,4.0,4.0,962.0,548.0,239882.241,15992.149400


## Step 3: check what's missing now

we already knew from notebook 01 that not every user in `train` shows up in `members` or `transactions`. now that everything's merged, let's see how that looks as missing values.

- `df.isna().mean() * 100` - same as before, % missing per column
- expecting `city`, `bd`, `gender`, etc to be missing for users not in `members`, and the transaction/log columns missing for users with no activity in the v2 files


In [4]:
(df.isna().mean() * 100).round(2)


msno                       0.00
is_churn                   0.00
city                      11.33
bd                        11.33
gender                    59.95
registered_via            11.33
registration_init_time    11.33
num_transactions           3.85
total_actual_paid          3.85
total_plan_list_price      3.85
is_cancel_sum              3.85
is_auto_renew_mean         3.85
last_transaction_date      3.85
last_expire_date           3.85
num_days                  22.29
num_25_sum                22.29
num_50_sum                22.29
num_75_sum                22.29
num_985_sum               22.29
num_100_sum               22.29
num_unq_sum               22.29
total_secs_sum            22.29
total_secs_mean           22.29
dtype: float64

## Step 4: fill in the transaction/log columns

for the columns that came from `transactions_agg` and `user_logs_agg`, a missing value just means "we didn't find any activity for this user" - which is basically the same as zero (0 transactions, 0 days listened, etc).

- `fillna(0)` - replaces all the `NaN`s in these columns with `0`
- leaving `last_transaction_date` and `last_expire_date` alone for now - those are dates, `0` doesn't really make sense there, so a missing date can just stay missing


In [5]:
fill_cols = [
    'num_transactions', 'total_actual_paid', 'total_plan_list_price',
    'is_cancel_sum', 'is_auto_renew_mean',
    'num_days', 'num_25_sum', 'num_50_sum', 'num_75_sum', 'num_985_sum',
    'num_100_sum', 'num_unq_sum', 'total_secs_sum', 'total_secs_mean',
]
df[fill_cols] = df[fill_cols].fillna(0)


## Step 5: clean up `bd` (age)

we already know `bd` has garbage in it - ages like `-7168` or `2016`. let's get rid of the obviously wrong ones by treating anything outside a normal human age range as missing.

- `df['bd'].between(0, 100)` - gives `True`/`False` for whether each value is in a sensible range
- `.where(condition)` - keeps the value where the condition is `True`, and replaces it with `NaN` where it's `False`
- so any age outside 0-100 just becomes `NaN` (unknown), instead of a weird number messing up the model later


In [6]:
df['bd'] = df['bd'].where(df['bd'].between(0, 100))
df['bd'].describe()


count    860444.000000
mean         13.440048
std          16.032548
min           0.000000
25%           0.000000
50%           0.000000
75%          27.000000
max         100.000000
Name: bd, dtype: float64

## Step 6: deal with `gender`

`gender` is missing for a lot of users (~65% back in notebook 01). instead of leaving it as `NaN`, let's just turn "missing" into its own category called `"unknown"`.

- `fillna('unknown')` - replaces `NaN` with the string `"unknown"`
- this way "we don't know this user's gender" becomes a real value the model can use, instead of a gap


In [7]:
df['gender'] = df['gender'].fillna('unknown')
df['gender'].value_counts(dropna=False)


gender
unknown    582055
male       204561
female     184344
Name: count, dtype: int64

## Step 7: final check and save

last look before saving - shape, any leftover missing values, then write it out.

- `df.shape` - should still be `(970960, ...)`, same number of rows as `train`
- `(df.isna().mean() * 100).round(2)` - check what (if anything) is still missing
- `to_parquet(...)` - save the final feature table so the modeling notebook can just load this directly


In [8]:
print(df.shape)
print((df.isna().mean() * 100).round(2))

df.to_parquet(PROCESSED / "features_v1.parquet")
print("saved:", PROCESSED / "features_v1.parquet")


(970960, 23)
msno                       0.00
is_churn                   0.00
city                      11.33
bd                        11.38
gender                     0.00
registered_via            11.33
registration_init_time    11.33
num_transactions           0.00
total_actual_paid          0.00
total_plan_list_price      0.00
is_cancel_sum              0.00
is_auto_renew_mean         0.00
last_transaction_date      3.85
last_expire_date           3.85
num_days                   0.00
num_25_sum                 0.00
num_50_sum                 0.00
num_75_sum                 0.00
num_985_sum                0.00
num_100_sum                0.00
num_unq_sum                0.00
total_secs_sum             0.00
total_secs_mean            0.00
dtype: float64
saved: ../data/processed/features_v1.parquet
